In [ ]:
import pandas as pd

df = pd.read_csv("results/imputation_results.csv")

df["method"] = pd.Categorical(df["method"], categories=["Simple", "KNN", "MissForest"], ordered=True)
df["split"] = pd.Categorical(df["split"], categories=["train", "test"], ordered=True)
df["missing_rate"] = pd.Categorical(df["missing_rate"], categories=["10%", "25%"], ordered=True)

metrics = ["rmse", "mae", "r2"]

def mean_and_empirical95CI(x: pd.Series):
    return pd.Series({
        "mean": x.mean(),
        "ci_lower_95": x.quantile(0.025),
        "ci_upper_95": x.quantile(0.975),
    })

summary = (
    df.groupby(["split", "missing_rate", "method"], observed=True)[metrics]
      .apply(lambda g: g.apply(mean_and_empirical95CI))
      .unstack(-1) 
)

summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index().sort_values(["split", "missing_rate", "method"]).reset_index(drop=True)

summary



In [ ]:
import pandas as pd

df = pd.read_csv("results/classification_results.csv")
df = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

for i in df["imputer"].unique():
    for j in sorted(df["missing_rate"].unique()):

        subset = df[
            (df["imputer"] == i) &
            (df["missing_rate"] == j)]

        if subset.empty:
            continue

        print(f"\n Imputation Method: {i} | Missing Rate: {j}")

        summary_table = (subset.groupby("classifier")[["accuracy", "f1", "roc_auc"]].mean().reset_index())
        print(summary_table)


In [ ]:
import pandas as pd

df = pd.read_csv("results/classification_results_complete.csv")

df_filtered = df[df["classifier"].isin(["LogisticRegression","XGBoost","NeuralNetwork"])]

print(f"\n Complete")

summary_table = (
    df_filtered
    .groupby("classifier")[["accuracy", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

print(summary_table)